# 08.4 — Latency Profiling

**Why profile?** NLP pipelines have many components — tokenization, embedding, retrieval, reranking, generation. You can't optimize what you haven't measured. Optimize the bottleneck, not the fast parts.

**What we'll build:** A profiler that measures time and memory for each pipeline stage, identifies bottlenecks, and reports P50/P95/P99 latencies.

---

In [ ]:
import time
import numpy as np
import functools
import tracemalloc
from contextlib import contextmanager
from collections import defaultdict
from typing import List, Callable, Dict, Any

# ---- Core profiling tools ----

class Timer:
    """High-resolution timer for profiling code blocks."""
    
    def __init__(self):
        self.records: Dict[str, List[float]] = defaultdict(list)
    
    @contextmanager
    def measure(self, name: str):
        start = time.perf_counter()
        yield
        elapsed = time.perf_counter() - start
        self.records[name].append(elapsed)
    
    def stats(self, name: str) -> Dict[str, float]:
        vals = np.array(self.records[name]) * 1000  # ms
        return {
            'n': len(vals),
            'mean_ms': float(np.mean(vals)),
            'p50_ms': float(np.percentile(vals, 50)),
            'p95_ms': float(np.percentile(vals, 95)),
            'p99_ms': float(np.percentile(vals, 99)),
            'total_ms': float(np.sum(vals)),
        }
    
    def report(self, sort_by='mean_ms'):
        all_stats = {name: self.stats(name) for name in self.records}
        sorted_items = sorted(all_stats.items(), key=lambda x: -x[1][sort_by])
        
        print(f"{'Stage':35} {'N':>5} {'Mean':>8} {'P50':>8} {'P95':>8} {'P99':>8} {'Total':>10}")
        print("-" * 85)
        total = sum(s['total_ms'] for s in all_stats.values())
        for name, s in sorted_items:
            share = 100 * s['total_ms'] / total if total > 0 else 0
            print(f"{name:35} {s['n']:>5} "
                  f"{s['mean_ms']:>7.2f}ms "
                  f"{s['p50_ms']:>7.2f}ms "
                  f"{s['p95_ms']:>7.2f}ms "
                  f"{s['p99_ms']:>7.2f}ms "
                  f"{s['total_ms']:>8.1f}ms ({share:.1f}%)")
        print(f"\nTotal wall time: {total:.1f}ms across {s['n']} iterations")


# Quick demo
timer = Timer()
for _ in range(20):
    with timer.measure("numpy_matmul"):
        a = np.random.randn(256, 256)
        b = np.random.randn(256, 256)
        c = a @ b
    with timer.measure("python_loop"):
        total = sum(range(10000))
    with timer.measure("list_comprehension"):
        result = [x**2 for x in range(1000)]

timer.report()

In [ ]:
# ---- Memory profiling ----

class MemoryProfiler:
    """Track peak memory usage of code blocks."""
    
    @contextmanager
    def measure(self, name: str = ""):
        tracemalloc.start()
        yield
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        print(f"{name or 'Block'}: peak memory = {peak / 1024 / 1024:.2f} MB")


mem_profiler = MemoryProfiler()

with mem_profiler.measure("Create large numpy array"):
    arr = np.random.randn(1000, 768).astype(np.float32)  # 1000 embeddings
    result = arr @ arr.T

with mem_profiler.measure("Python list of dicts"):
    data = [{'id': i, 'text': f'document {i}' * 10} for i in range(10000)]

print()
print("Memory estimation formulas:")
n_docs = 1_000_000
d = 768
dtype_bytes = 4  # float32
print(f"  1M embeddings (d={d}, float32): {n_docs * d * dtype_bytes / 1024**3:.2f} GB")
print(f"  1M embeddings (d={d}, float16): {n_docs * d * 2 / 1024**3:.2f} GB")
print(f"  1M embeddings (d=384, float32): {n_docs * 384 * 4 / 1024**3:.2f} GB")
print()
print("Tip: Use float16 for inference-only embeddings. Saves 50% memory, minimal quality loss.")

In [ ]:
# ---- Profile a complete RAG pipeline ----

import re
from collections import Counter
import math

# Simulate pipeline components with realistic timing

def simulate_tokenize(text: str) -> list:
    time.sleep(np.random.uniform(0.0001, 0.0005))
    return re.findall(r'\b[a-z]+\b', text.lower())


def simulate_embed_query(tokens: list) -> np.ndarray:
    # Simulates a small neural encoder forward pass
    time.sleep(np.random.uniform(0.002, 0.008))
    vec = np.random.randn(384).astype(np.float32)
    return vec / np.linalg.norm(vec)


def simulate_faiss_search(query_emb: np.ndarray, index: np.ndarray, k=5) -> list:
    # Simulates FAISS flat IP search
    time.sleep(np.random.uniform(0.001, 0.003))
    scores = index @ query_emb
    top_k = np.argsort(-scores)[:k]
    return top_k.tolist()


def simulate_rerank(query: str, passages: list, k=3) -> list:
    # Cross-encoder reranker is the slow step
    time.sleep(np.random.uniform(0.015, 0.040) * len(passages))
    return passages[:k]


def simulate_llm_generate(prompt: str) -> str:
    # LLM generation — typically 100-500ms for small models
    n_tokens = len(prompt.split()) // 4 + 50  # rough estimate
    time.sleep(np.random.uniform(0.050, 0.200))
    return f"[Generated response of ~{n_tokens} tokens]"


# Build index
N_DOCS = 10000
D = 384
print(f"Building index with {N_DOCS} documents...")
index = np.random.randn(N_DOCS, D).astype(np.float32)
norms = np.linalg.norm(index, axis=1, keepdims=True)
index /= norms
passages = [f"Passage {i}: content about topic {i % 20}" for i in range(N_DOCS)]
print("Index ready.")


# Profile the full pipeline
timer = Timer()
N_QUERIES = 30
queries = [f"What is {topic}?" for topic in ["BERT", "attention", "RAG", "LoRA"] * 8]

print(f"\nProfiling {N_QUERIES} queries...")
for query in queries[:N_QUERIES]:
    with timer.measure("1_tokenize"):
        tokens = simulate_tokenize(query)
    
    with timer.measure("2_embed_query"):
        q_emb = simulate_embed_query(tokens)
    
    with timer.measure("3_faiss_retrieve"):
        top_ids = simulate_faiss_search(q_emb, index, k=10)
    
    retrieved_passages = [passages[i] for i in top_ids]
    
    with timer.measure("4_rerank"):
        reranked = simulate_rerank(query, retrieved_passages, k=3)
    
    prompt = f"Context: {' '.join(reranked)}\n\nQuestion: {query}\n\nAnswer:"
    
    with timer.measure("5_llm_generate"):
        answer = simulate_llm_generate(prompt)

print()
timer.report()

In [ ]:
# ---- Identifying and fixing bottlenecks ----

print("=== Bottleneck Analysis ===")
print()
print("From the profile above, the bottlenecks are typically:")
print("  1. LLM generation (50-200ms per query) — the #1 bottleneck")
print("  2. Cross-encoder reranking (15-40ms × n_passages)")
print("  3. Embedding the query (2-8ms for transformer encoder)")
print("  4. FAISS search (1-3ms even for 10M vectors — not a bottleneck)")
print("  5. Tokenization (< 1ms — never a bottleneck)")
print()
print("=== Optimization techniques ===")
print()

optimizations = [
    ("Batching",
     "Process multiple queries together. GPU utilization goes from 10% to 80%+.",
     "Increases latency for individual queries; great for throughput."),
    
    ("KV-cache",
     "For LLMs: cache key-value pairs from the system prompt across requests.",
     "Reduces prefill cost when system prompt is shared. Claude API supports this."),
    
    ("Speculative decoding",
     "Small draft model generates tokens; large model verifies in parallel.",
     "2-3× speedup for LLM generation with no quality loss."),
    
    ("fp16/int8 quantization",
     "Cut model size in half (fp16) or quarter (int8). Minimal quality loss.",
     "INT8: use bitsandbytes or GPTQ. FP16: just model.half()."),
    
    ("Embedding caching",
     "Cache embeddings for repeated queries (exact or near-duplicate).",
     "Works well for FAQ-style systems. Use Redis or in-memory LRU."),
    
    ("Async retrieval",
     "Start embedding query while previous query is still being generated.",
     "Overlap I/O and compute. Use asyncio."),
    
    ("Smaller embedding model",
     "all-MiniLM-L6-v2 (22M params) is 5× faster than all-mpnet-base (110M).",
     "Quality tradeoff; profile quality/latency Pareto curve."),
]

for name, benefit, note in optimizations:
    print(f"  {name:25}: {benefit}")
    print(f"  {'':25}  Note: {note}")
    print()

In [ ]:
# ---- P50/P95/P99: why percentiles matter more than averages ----

print("=== Why percentiles > averages for latency ===")
print()

# Simulate bimodal latency (fast + occasional slow responses)
np.random.seed(42)
fast_requests = np.random.normal(50, 5, 950)   # 950 fast requests: ~50ms
slow_requests = np.random.normal(500, 50, 50)  # 50 slow requests: ~500ms (cache miss)
latencies = np.concatenate([fast_requests, slow_requests])

print(f"Distribution: 950 fast (~50ms) + 50 slow (~500ms)")
print(f"  Mean:   {np.mean(latencies):.1f} ms  <- misleading (inflated by 5% slow)")
print(f"  Median: {np.median(latencies):.1f} ms  <- represents typical user")
print(f"  P95:    {np.percentile(latencies, 95):.1f} ms  <- 1 in 20 users waits this long")
print(f"  P99:    {np.percentile(latencies, 99):.1f} ms  <- 1 in 100 users (SLA threshold)")
print(f"  Max:    {np.max(latencies):.1f} ms")
print()
print("SLA definition: 'P99 < 500ms' is meaningful. 'Average < 100ms' can hide problems.")
print()
print("Latency budget for a RAG API (example target: P99 < 2000ms):")
budget = [
    ("Query embedding",    "< 20ms",  "Use fast model (MiniLM), batch where possible"),
    ("FAISS retrieval",    "< 10ms",  "Flat index for <1M docs, IVF for larger"),
    ("Reranker",           "< 100ms", "Skip for low-latency; only top-10 candidates"),
    ("LLM generation",     "< 1500ms","Stream output; use smaller model if OK"),
    ("Overhead/network",   "< 100ms", "Connection pooling, keep-alive"),
    ("Buffer",             "< 270ms", "For variance"),
]
total = 0
for stage, target, tip in budget:
    ms = int(target.replace('< ', '').replace('ms', ''))
    total += ms
    print(f"  {stage:20}: {target:10} — {tip}")
print(f"  {'TOTAL':20}: {total}ms")

## Summary

**Profile before optimizing.** In RAG pipelines:
- LLM generation = 60-80% of latency
- Cross-encoder reranking = 10-20%
- Everything else = fast

**Report P95/P99, not mean.** Users experience worst-case latency, not average.

**Optimization priority:**
1. Smaller/quantized LLM or streaming (biggest impact)
2. Skip reranker when not needed
3. Cache embeddings for repeated queries
4. Async/batched processing for throughput

---
**Next:** 08.5 — Model Serving (vLLM, llama.cpp, quantization)